<a href="https://colab.research.google.com/github/john-dechellis-weather/wx_compare/blob/main/RadarLocation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install metpy siphon cartopy ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 424.4/424.4 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.1/65.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.5/307.5 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 69.6 MB/s eta 0:00:00


In [ ]:
from datetime import datetime
from io import BytesIO
from urllib.request import urlopen

import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import ipywidgets as widgets
from IPython.display import display, clear_output

from siphon.radarserver import RadarServer, get_radarserver_datasets
from metpy.io import Level3File
from metpy.calc import azimuth_range_to_lat_lon
from metpy.plots import colortables
from metpy.units import units

In [ ]:
base_server = "https://thredds.ucar.edu/thredds/"

def get_radar_server():
    datasets = get_radarserver_datasets(base_server)
    radar_ref = datasets["NEXRAD Level III Radar from IDD"]
    return RadarServer(radar_ref.follow().catalog_url)

rs = get_radar_server()


def plot_radar(target_time, aircraft_lat, aircraft_lon, station="ILX", product="N0B", zoom_deg=2.0, label_text="Location"):
    query = rs.query()
    query.stations(station).time(target_time).variables(product)

    catalog = rs.get_catalog(query)
    matches = list(catalog.datasets.values())

    if not matches:
        raise ValueError("No radar dataset returned for that time/station/product.")

    dataset = matches[0]
    print("Dataset:", dataset.name)

    nids_url = dataset.access_urls["HTTPServer"]
    with urlopen(nids_url) as resp:
        raw = resp.read()

    f = Level3File(BytesIO(raw))

    print("Radar center lat/lon:", f.lat, f.lon)
    print("Max range (km):", f.max_range)

    datadict = f.sym_block[0][0]
    radar_data = f.map_data(datadict["data"])

    az = units.Quantity(
        np.array(datadict["start_az"] + [datadict["end_az"][-1]]),
        "degrees"
    )
    rng = units.Quantity(
        np.linspace(0, f.max_range, radar_data.shape[-1] + 1),
        "kilometers"
    )

    lon_grid, lat_grid = azimuth_range_to_lat_lon(az, rng, f.lon, f.lat)

    fig = plt.figure(figsize=(12, 10))
    ax = plt.axes(projection=ccrs.PlateCarree())

    norm, cmap = colortables.get_with_steps(
        "NWSStormClearReflectivity", -20, 0.5
    )

    mesh = ax.pcolormesh(
        lon_grid,
        lat_grid,
        radar_data,
        cmap=cmap,
        norm=norm,
        shading="auto",
        transform=ccrs.PlateCarree()
    )

    ax.coastlines(resolution="10m", color="black", linewidth=0.8)
    ax.add_feature(cfeature.BORDERS.with_scale("10m"), edgecolor="black", linewidth=0.6)
    ax.add_feature(cfeature.STATES.with_scale("10m"), edgecolor="black", linewidth=0.5, facecolor="none")

    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        linewidth=0.6,
        color="gray",
        alpha=0.7,
        linestyle="--"
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 9}
    gl.ylabel_style = {"size": 9}

    ax.scatter(
        aircraft_lon,
        aircraft_lat,
        s=180,
        marker="x",
        color="red",
        zorder=10,
        transform=ccrs.PlateCarree()
    )

    ax.text(
        aircraft_lon + 0.05,
        aircraft_lat + 0.05,
        label_text,
        color="red",
        fontsize=12,
        zorder=10,
        transform=ccrs.PlateCarree()
    )

    ax.set_extent(
        [aircraft_lon - zoom_deg, aircraft_lon + zoom_deg,
         aircraft_lat - zoom_deg, aircraft_lat + zoom_deg],
        crs=ccrs.PlateCarree()
    )

    ax.set_title(f"{station} {product} Radar Reflectivity with Position\n{target_time:%Y-%m-%d %H:%M UTC}")
    plt.colorbar(mesh, ax=ax, pad=0.02, label="dBZ")
    plt.show()

In [ ]:
station_widget = widgets.Text(value="FTG", description="Station:")
product_widget = widgets.Dropdown(
    options=["N0B", "N0Q", "N0R", "N1P", "NTP"],
    value="N0B",
    description="Product:"
)

lat_widget = widgets.FloatText(value=39.76, description="Lat:")
lon_widget = widgets.FloatText(value=-86.15, description="Lon:")

date_widget = widgets.Text(value="2026-07-06", description="Date:")
time_widget = widgets.Text(value="23:30", description="Time UTC:")

zoom_widget = widgets.FloatSlider(
    value=2.0, min=0.5, max=5.0, step=0.5, description="Zoom:"
)

label_widget = widgets.Text(value="Location", description="Label:")
run_button = widgets.Button(description="Render Radar", button_style="primary")
output = widgets.Output()

def run_plot(_):
    with output:
        clear_output(wait=True)
        try:
            target_time = datetime.strptime(
                f"{date_widget.value} {time_widget.value}",
                "%Y-%m-%d %H:%M"
            )
            plot_radar(
                target_time=target_time,
                aircraft_lat=lat_widget.value,
                aircraft_lon=lon_widget.value,
                station=station_widget.value.strip().upper(),
                product=product_widget.value,
                zoom_deg=zoom_widget.value,
                label_text=label_widget.value
            )
        except Exception as e:
            print("Error:", e)

run_button.on_click(run_plot)

display(
    widgets.VBox([
        station_widget,
        product_widget,
        lat_widget,
        lon_widget,
        date_widget,
        time_widget,
        zoom_widget,
        label_widget,
        run_button,
        output
    ])
)